# 🗄️ Step 2 of RAG — Store Embeddings in PostgreSQL

In the previous notebook we read, chunked, and embedded *Alice's Adventures in Wonderland*. Those embeddings lived only in memory — useless the moment the kernel restarts.

This notebook persists them to a **vector database** so they can be queried efficiently at any time:

```
Chunks + Embeddings  ──►  PostgreSQL + pgvector  ──►  Persistent Vector Store
```

> 💡 **Why PostgreSQL + pgvector instead of a dedicated vector DB?**  
> Tools like Pinecone or Weaviate are purpose-built for vectors, but they add infrastructure complexity and cost. pgvector is a PostgreSQL extension that adds vector similarity search to a database you may already be running. For most RAG projects — including production ones — it is fast enough and far simpler to operate. Supabase gives us a managed PostgreSQL instance with pgvector pre-installed, so we get a vector database with zero extra setup.

## 0. Setup & Imports

In [ ]:
import os
import re
import json
import numpy as np
import psycopg2
from pathlib import Path
from dotenv import load_dotenv
from munch import Munch
from FlagEmbedding import BGEM3FlagModel

### Load config & secrets

In [ ]:
# 📂 Load config
config_path = (Path.cwd() / "../config/config.yaml").resolve()
with open(config_path, "r") as f:
    config = Munch.fromYAML(f)

# 🔑 Load secrets from .env
load_dotenv((Path.cwd() / "../.env").resolve())
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")

if not SUPABASE_PASSWORD:
    raise ValueError("SUPABASE_PASSWORD not found in .env file")

print("✅ Config loaded")
print(f"✅ SUPABASE_PASSWORD found")

---

## 1. Connect to the Database

We build the connection URL from `config.yaml` (host, port, database, user) and the password from `.env`. Keeping the password out of config files means it never accidentally ends up in version control.

In [ ]:
DATABASE_URL = (
    f"postgresql://{config.supabase.user}:{SUPABASE_PASSWORD}@"
    f"{config.supabase.host}:{config.supabase.port}/{config.supabase.database}"
)

def get_connection():
    """Return a new psycopg2 connection."""
    return psycopg2.connect(DATABASE_URL)

# Smoke test
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT version();")
print(f"✅ Connected — {cur.fetchone()[0][:60]}...")
cur.close()
conn.close()

---

## 2. Enable pgvector & Create Tables

### Why three separate tables?

Each embedding type has a fundamentally different shape:

| Table | Embedding | PostgreSQL type | Shape per chunk |
|---|---|---|---|
| `embeddings_dense` | Dense | `vector(1024)` | Fixed 1024-dim float array |
| `embeddings_sparse` | Sparse (lexical) | `jsonb` | Variable-length `{token_id: weight}` dict |
| `embeddings_colbert` | ColBERT | `vector(1024)[]` | Array of 1024-dim vectors, one per token |

Mixing them into one table would force awkward NULLs and make queries harder to read. Separate tables keep the schema honest.

The SQL for each table lives in `src/sql/` so you can inspect, version-control, and re-run them independently.

In [ ]:
def run_sql_file(path: Path) -> None:
    """Execute a .sql file against the database."""
    sql = path.read_text()
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute(sql)
        conn.commit()
        print(f"✅ Executed {path.name}")
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        cur.close()
        conn.close()


sql_dir = (Path.cwd() / "sql").resolve()

run_sql_file(sql_dir / "01_enable_pgvector.sql")
run_sql_file(sql_dir / "02_create_table_dense.sql")
run_sql_file(sql_dir / "03_create_table_sparse.sql")
run_sql_file(sql_dir / "04_create_table_colbert.sql")

---

## 3. Prepare Chunks & Embeddings

We reuse the same pipeline from Notebook 2 — read, clean, chunk, embed. This is deliberately brief here since the focus of this notebook is storage, not the embedding process itself.

In [ ]:
# ── Helpers from Notebook 2 ───────────────────────────────────

def clean_text(text: str) -> str:
    text = re.sub(r"\[Illustration[^\]]*\]", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = "\n".join(line.strip() for line in text.splitlines())
    return text.strip()


def paragraph_chunks(text: str, min_chars: int) -> list[str]:
    """
    Split text on blank lines; discard very short paragraphs.

    Args:
        text      : The input text.
        min_chars : Minimum character count to keep a paragraph.

    Returns:
        List of chunk strings.
    """
    paragraphs = re.split(r"\n{2,}", text)
    return [p.strip() for p in paragraphs if len(p.strip()) >= min_chars]


# ── Load & chunk the book ─────────────────────────────────────
doc_path = (Path.cwd() / "../assets" / config.chunking.book_file).resolve()
raw_text = doc_path.read_text(encoding="utf-8")
clean = clean_text(raw_text)

# Use paragraph chunking instead of fixed-size
min_chars = config.chunking.paragraph.min_chars
chunks = paragraph_chunks(clean, min_chars=min_chars)

# Print statistics
print(f"Settings  : min_chars={min_chars}")
print(f"Chunks    : {len(chunks)}")
print(f"Avg length: {sum(len(c) for c in chunks) / len(chunks):.0f} chars")
print(f"Min length: {min(len(c) for c in chunks)} chars")
print(f"Max length: {max(len(c) for c in chunks)} chars")
print(f"\n--- Example chunk  ---")
print(f"\n✅ {len(chunks)} chunks ready")


In [ ]:
# ── Configuration ─────────────────────────────────────────────
NUM_CHUNKS_TO_EMBED = config.embedding.num_chunks_embed  # Set to None to embed all chunks, or specify a number

# ── Embed all chunks ──────────────────────────────────────────
print(f"Loading {config.embedding.model}...")
model = BGEM3FlagModel(config.embedding.model, use_fp16=True)

# Select chunks to embed
chunks = chunks[:NUM_CHUNKS_TO_EMBED] if NUM_CHUNKS_TO_EMBED else chunks

print(f"Embedding {len(chunks)} chunks — this may take a minute...")
embeddings = model.encode(
    chunks,
    batch_size=config.embedding.batch_size,
    max_length=config.embedding.max_length,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
)

dense_vecs   = embeddings["dense_vecs"]       # np.ndarray (N, 1024)
sparse_vecs  = embeddings["lexical_weights"]  # list of dicts
colbert_vecs = embeddings["colbert_vecs"]     # list of np.ndarray (n_tokens, 1024)

print(f"✅ Embeddings ready — dense shape: {dense_vecs.shape}")


---

## 4. Store Dense Embeddings

Dense vectors are stored as `vector(1024)` — a native pgvector type. PostgreSQL stores them as a compact binary array and can compute cosine similarity directly in SQL with the `<=>` operator.

We also store the HNSW index definition in the SQL file. HNSW (Hierarchical Navigable Small World) is an approximate nearest-neighbour algorithm that makes similarity search fast even over millions of vectors.

In [ ]:
def store_dense(chunks: list, dense_vecs, sql: str) -> None:
    """Insert all chunks + dense vectors into embeddings_dense."""
    from psycopg2.extras import execute_values
    records = [
        (i, chunks[i], dense_vecs[i].tolist())
        for i in range(len(chunks))
    ]
    conn = get_connection()
    cur = conn.cursor()
    try:
        execute_values(cur, sql, records)
        conn.commit()
        print(f"✅ Stored {len(chunks)} dense vectors")
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        cur.close()
        conn.close()


sql_insert_dense = (sql_dir / "05_insert_dense.sql").read_text()
store_dense(chunks, dense_vecs, sql_insert_dense)


---

## 5. Store Sparse Embeddings

Sparse embeddings are dictionaries of `{token_id: weight}`. Most token IDs are absent for any given chunk — storing them as a dense vector would waste enormous space. Instead we use `jsonb`, PostgreSQL's binary JSON type, which:

- Stores only the non-zero entries
- Supports GIN indexing for fast key lookups
- Is natively queryable with SQL

> Note: token IDs from `bge-m3` are integers, but Python dicts with integer keys serialise to JSON with string keys. We store them as strings and convert back at query time.

In [ ]:
def store_sparse(chunks: list, sparse_vecs: list, sql: str) -> None:
    """Insert all chunks + sparse dicts into embeddings_sparse."""
    from psycopg2.extras import execute_values
    records = [
        (i, chunks[i], json.dumps({str(k): float(v) for k, v in sparse_vecs[i].items()}))
        for i in range(len(chunks))
    ]
    conn = get_connection()
    cur = conn.cursor()
    try:
        execute_values(cur, sql, records)
        conn.commit()
        print(f"✅ Stored {len(chunks)} sparse vectors")
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        cur.close()
        conn.close()


sql_insert_sparse = (sql_dir / "06_insert_sparse.sql").read_text()
store_sparse(chunks, sparse_vecs, sql_insert_sparse)


---

## 6. Store ColBERT Embeddings

ColBERT produces one vector **per token**, so each chunk maps to a 2D matrix `(n_tokens, 1024)` rather than a single vector. We store this as `vector(1024)[]` — a PostgreSQL array of vectors.

This is more storage-intensive than dense (roughly 30–80× more vectors to store), which is why ColBERT is typically used as a **re-ranker** over a shortlist from dense/sparse retrieval rather than a first-pass retriever over the entire corpus.

In [ ]:
def store_colbert(chunks: list, colbert_vecs: list, sql: str) -> None:
    from psycopg2.extras import execute_values
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm import tqdm

    # Pre-format vectors as pgvector strings in Python
    records = [
        (chunk_id, token_idx, chunks[chunk_id], f"[{','.join(map(str, token_vec.tolist()))}]")
        for chunk_id, token_matrix in enumerate(colbert_vecs)
        for token_idx, token_vec in enumerate(token_matrix)
    ]

    total_tokens = len(records)
    batch_size = 100
    batches = [records[i : i + batch_size] for i in range(0, total_tokens, batch_size)]
    print(f"  {total_tokens:,} token rows — {len(batches)} batches of {batch_size}")

    def insert_batch(batch):
        conn = get_connection()
        cur = conn.cursor()
        try:
            execute_values(cur, sql, batch, page_size=batch_size)
            conn.commit()
        finally:
            cur.close()
            conn.close()

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(insert_batch, b): i for i, b in enumerate(batches)}
        with tqdm(total=len(batches), desc="Storing ColBERT") as pbar:
            for future in as_completed(futures):
                future.result()  # raises immediately if a batch failed
                pbar.update(1)

    print(f"✅ Stored {total_tokens:,} ColBERT token vectors across {len(chunks)} chunks")

sql_insert_colbert = (sql_dir / "07_insert_colbert.sql").read_text()
store_colbert(chunks, colbert_vecs, sql_insert_colbert)

In [ ]:
# Create index with extended timeout in the same connection
sql_index = (sql_dir / "08_create_index_colbert.sql").read_text()

conn = get_connection()
cur = conn.cursor()
try:
    # Set timeout and create index in the same connection
    cur.execute("SET statement_timeout = '60min';")
    print("✅ Extended statement timeout to 60 minutes")
    
    cur.execute(sql_index)
    conn.commit()
    print("✅ Executed 08_create_index_colbert.sql")
except Exception as e:
    conn.rollback()
    raise e
finally:
    cur.close()
    conn.close()

---

## 7. Verify — Row Counts & Sample

A quick sanity check: all three tables should have the same number of rows as we have chunks.

In [ ]:
conn = get_connection()
cur = conn.cursor()

for table in ["embeddings_dense", "embeddings_sparse", "embeddings_colbert"]:
    cur.execute(f"SELECT COUNT(*) FROM {table};")
    count = cur.fetchone()[0]
    status = "✅" if count == len(chunks) else "⚠️"
    print(f"{status} {table:<25} {count:>5} rows  (expected {len(chunks)})")

# Peek at one dense row
print("\n--- Sample dense row ---")
cur.execute("SELECT chunk_id, LEFT(chunk_text, 80), embedding::text FROM embeddings_dense LIMIT 1;")
row = cur.fetchone()
print(f"chunk_id  : {row[0]}")
print(f"chunk_text: {row[1]}...")
print(f"embedding : {row[2][:60]}...")

cur.close()
conn.close()

---

## 8. Summary & Key Takeaways

| Topic | What we learned |
|---|---|
| **pgvector** | A PostgreSQL extension that adds native vector types and similarity operators |
| **Why not a dedicated vector DB?** | pgvector is simpler to operate and fast enough for most RAG workloads |
| **Dense table** | `vector(1024)` column; supports HNSW index for fast approximate search |
| **Sparse table** | `jsonb` column; stores only non-zero token weights efficiently |
| **ColBERT table** | `jsonb` matrix per chunk; more storage but highest precision at re-ranking |

### What comes next?

In the next notebook we'll query these tables — running dense, sparse, and ColBERT searches against a user question and comparing what each retrieves.